# ATARI 


## set up local environment

In [2]:
print("hi from atari nb")

hi from atari nb


In [3]:
from __future__ import absolute_import, division, print_function

import base64
import imageio
import IPython  
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import pyvirtualdisplay
import reverb

import tensorflow as tf
import gym # 0.26.1 is required

# To get smooth animations
import matplotlib.animation as animation
matplotlib.rc('animation', html='jshtml')

from tf_agents.agents.dqn import dqn_agent
from tf_agents.drivers import py_driver
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.eval import metric_utils
from tf_agents.metrics import tf_metrics
from tf_agents.networks import sequential
from tf_agents.policies import py_tf_eager_policy
from tf_agents.policies import random_tf_policy
from tf_agents.replay_buffers import reverb_replay_buffer
from tf_agents.replay_buffers import reverb_utils
from tf_agents.trajectories import trajectory
from tf_agents.specs import tensor_spec
from tf_agents.utils import common

from tf_agents.environments import  suite_gym
from tf_agents.environments.atari_preprocessing import AtariPreprocessing
from tf_agents.environments.atari_wrappers import FrameStack4
from tf_agents.environments import suite_gym, tf_py_environment
from tf_agents.networks import q_network
from tf_agents.agents.dqn import dqn_agent
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.trajectories import trajectory
from tf_agents.utils import common
from tf_agents.drivers import dynamic_step_driver
from tf_agents.policies import random_tf_policy
import gym

# To get smooth animations
import matplotlib.animation as animation
matplotlib.rc('animation', html='jshtml')



/home/macreat/miniforge3/envs/RLEnv/lib/python3.11/site-packages/imageio/core/util.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


2025-12-02 15:18:02.038215: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-02 15:18:02.292557: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-02 15:18:02.292714: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-02 15:18:02.292887: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-02 15:18:02.344784: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-02 15:18:02.349642: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

## entornos de ATARI

definimos el entorno PONG, seleccionando una version compatible con tf-agents 

In [4]:
gym.envs.registry

├──ALE
│   ├──ALE/Adventure: [ v5 ]
│   ├──ALE/Adventure-ram: [ v5 ]
│   ├──ALE/AirRaid: [ v5 ]
│   ├──ALE/AirRaid-ram: [ v5 ]
│   ├──ALE/Alien: [ v5 ]
│   ├──ALE/Alien-ram: [ v5 ]
│   ├──ALE/Amidar: [ v5 ]
│   ├──ALE/Amidar-ram: [ v5 ]
│   ├──ALE/Assault: [ v5 ]
│   ├──ALE/Assault-ram: [ v5 ]
│   ├──ALE/Asterix: [ v5 ]
│   ├──ALE/Asterix-ram: [ v5 ]
│   ├──ALE/Asteroids: [ v5 ]
│   ├──ALE/Asteroids-ram: [ v5 ]
│   ├──ALE/Atlantis: [ v5 ]
│   ├──ALE/Atlantis-ram: [ v5 ]
│   ├──ALE/Atlantis2: [ v5 ]
│   ├──ALE/Atlantis2-ram: [ v5 ]
│   ├──ALE/Backgammon: [ v5 ]
│   ├──ALE/Backgammon-ram: [ v5 ]
│   ├──ALE/BankHeist: [ v5 ]
│   ├──ALE/BankHeist-ram: [ v5 ]
│   ├──ALE/BasicMath: [ v5 ]
│   ├──ALE/BasicMath-ram: [ v5 ]
│   ├──ALE/BattleZone: [ v5 ]
│   ├──ALE/BattleZone-ram: [ v5 ]
│   ├──ALE/BeamRider: [ v5 ]
│   ├──ALE/BeamRider-ram: [ v5 ]
│   ├──ALE/Berzerk: [ v5 ]
│   ├──ALE/Berzerk-ram: [ v5 ]
│   ├──ALE/Blackjack: [ v5 ]
│   ├──ALE/Blackjack-ram: [ v5 ]
│   ├──ALE/Bowling: [ v5 ]
│ 

In [5]:
env_name = "ALE/Pong-v5"   # versión más compatible con TF-Agents

train_env = tf_py_environment.TFPyEnvironment(
    suite_gym.load(env_name)
)

eval_env = tf_py_environment.TFPyEnvironment(
    suite_gym.load(env_name)
)


A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


## Diagnose environment observation spec

In [ ]:
print("Train env observation_spec():")
print(train_env.observation_spec())
print("\nTrain env batch_size:", train_env.batch_size)
print("\nAction spec:")
print(train_env.action_spec())

# Test reset to see actual observation shape
print("\n--- Testing reset ---")
try:
    ts = train_env.reset()
    print("Reset successful. Observation shape:", ts.observation.shape, "dtype:", ts.observation.dtype)
except Exception as e:
    print("Reset failed with error:")
    print(type(e).__name__, ":", str(e)[:200])

Train env observation_spec():
BoundedTensorSpec(shape=(210, 160, 3), dtype=tf.uint8, name='observation', minimum=array(0, dtype=uint8), maximum=array(255, dtype=uint8))

Train env batch_size: 1

Action spec:
BoundedTensorSpec(shape=(), dtype=tf.int64, name='action', minimum=array(0), maximum=array(5))

--- Testing reset ---
Reset failed with error:
ValueError : setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


In [7]:
# Recreate environments: Use suite_gym with AtariPreprocessing for Pong
from tf_agents.environments import suite_atari

# Load Atari Pong with built-in preprocessing
# This handles frame stacking and preprocessing correctly
train_py_env = suite_atari.load(
    "PongNoFrameskip-v4",
    max_episode_steps=10000,
    gym_env_wrappers=[],  # No additional wrappers
)

eval_py_env = suite_atari.load(
    "PongNoFrameskip-v4",
    max_episode_steps=10000,
    gym_env_wrappers=[],
)

# Wrap with TFPyEnvironment
train_env = tf_py_environment.TFPyEnvironment(train_py_env)
eval_env = tf_py_environment.TFPyEnvironment(eval_py_env)

print("Environment loaded successfully with suite_atari.")
print("Observation spec:", train_env.observation_spec())
print("Action spec:", train_env.action_spec())

# Test reset
try:
    ts = train_env.reset()
    print("\nReset successful!")
    print("Observation shape:", ts.observation.shape)
    print("Observation dtype:", ts.observation.dtype)
except Exception as e:
    print(f"\nReset failed: {type(e).__name__}: {e}")

Environment loaded successfully with suite_atari.
Observation spec: BoundedTensorSpec(shape=(210, 160, 3), dtype=tf.uint8, name='observation', minimum=array(0, dtype=uint8), maximum=array(255, dtype=uint8))
Action spec: BoundedTensorSpec(shape=(), dtype=tf.int64, name='action', minimum=array(0), maximum=array(5))

Reset failed: ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


In [8]:
# Limit GPU memory growth to avoid allocation errors
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"GPU memory growth setting error: {e}")

# Force CPU-only mode for safety on limited memory systems
tf.config.set_visible_devices([], 'GPU')
print("GPU disabled. Running on CPU only.")

GPU disabled. Running on CPU only.


2025-12-02 15:18:28.260544: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-12-02 15:18:28.264035: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## creamos una red Q (CNN mínima para entornos de ATARI)

In [9]:
fc_layer_params = (256,)

# Create a custom Q-network with preprocessing
    
from tf_agents.networks import network as network_module

class AtariQNetwork(network_module.Network):
    def __init__(self, observation_spec, action_spec, conv_layer_params,
                 fc_layer_params, name='AtariQNetwork'):
        super(AtariQNetwork, self).__init__(input_tensor_spec=observation_spec,
                                             state_spec=(), name=name)
        # Normalization layer to convert uint8 to float32 in [0, 1]
        self._normalization = tf.keras.layers.Lambda(
            lambda x: tf.cast(x, tf.float32) / 255.0
        )
        # Create a float32 version of the observation spec for the internal QNetwork
        def _to_float32_spec(spec):
            try:
                return spec.replace(dtype=tf.float32)
            except Exception:
                return tensor_spec.TensorSpec(spec.shape, dtype=tf.float32)

        float_observation_spec = tf.nest.map_structure(_to_float32_spec, observation_spec)

        self._q_net = q_network.QNetwork(
            float_observation_spec,
            action_spec,
            conv_layer_params=conv_layer_params,
            fc_layer_params=fc_layer_params,
        )

    def call(self, observation, step_type=None, network_state=(), training=False):
        # Preprocess: normalize to [0, 1]
        normalized_observation = self._normalization(observation)
        # Delegate to internal QNetwork
        return self._q_net(
            normalized_observation,
            step_type=step_type,
            network_state=network_state,
            training=training,
        )

q_net = AtariQNetwork(
    train_env.observation_spec(),
    train_env.action_spec(),
    conv_layer_params=[(32, 8, 4), (64, 4, 2), (64, 3, 1)],
    fc_layer_params=fc_layer_params
)


## y creamos el agete DQN para este caso

In [10]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

train_step_counter = tf.Variable(0)

agent = dqn_agent.DqnAgent(
    train_env.time_step_spec(),
    train_env.action_spec(),
    q_network=q_net,
    optimizer=optimizer,
    td_errors_loss_fn=common.element_wise_squared_loss,
    train_step_counter=train_step_counter,
    epsilon_greedy=0.1
)

agent.initialize()


## ahora, definimos el repy buffer 

In [11]:
# PRE-TRAINING DIAGNOSTIC: Verify all required components
print("=" * 60)
print("PRE-TRAINING DIAGNOSTIC")
print("=" * 60)

required_vars = {
    'train_env': 'Training environment',
    'eval_env': 'Evaluation environment',
    'q_net': 'Q-Network',
    'agent': 'DQN Agent',
    'optimizer': 'Optimizer',
}

missing = []
for var_name, description in required_vars.items():
    try:
        val = eval(var_name)
        print(f"✓ {description:30s} ({var_name}): OK")
    except NameError:
        print(f"✗ {description:30s} ({var_name}): MISSING")
        missing.append(var_name)

print("=" * 60)
if missing:
    print(f"\nERROR: Missing components: {', '.join(missing)}")
    print("\nPlease run these cells in order:")
    print("1. Imports (cell 3)")
    print("2. Environment setup (cell 6: suite_gym version)")
    print("3. Environment recreation (cell 9: GymWrapper version)")
    print("4. GPU disable (cell 10)")
    print("5. Q-Network (cell 11)")
    print("6. Agent (cell 12)")
    print("\nThen you can run this diagnostic again.\n")
else:
    print("\n✓ All components ready! Proceeding to replay buffer...\n")

PRE-TRAINING DIAGNOSTIC
✓ Training environment           (train_env): OK
✓ Evaluation environment         (eval_env): OK
✓ Q-Network                      (q_net): OK
✓ DQN Agent                      (agent): OK
✓ Optimizer                      (optimizer): OK

✓ All components ready! Proceeding to replay buffer...



## procedemos el entreno y evaluación

In [13]:
# Define TF-Uniform Replay Buffer (MINIMAL for CPU-only with limited memory)
replay_buffer_capacity = 5000  # DRASTICALLY REDUCED from 50000 to ~125MB
# Ensure agent exists before creating replay buffer
try:
    data_spec = agent.collect_data_spec
except NameError:
    raise RuntimeError('Agent not defined. Run the DqnAgent creation cell before this one.')

batch_size = getattr(train_env, 'batch_size', 1)
replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=data_spec,
    batch_size=batch_size,
    max_length=replay_buffer_capacity,
)

# Dataset for sampling from the replay buffer (MINIMAL: no prefetch, no parallelism)
dataset = replay_buffer.as_dataset(
    sample_batch_size=16,  # Further reduced from 32
    num_steps=2,  # Reduced back to 2
    num_parallel_calls=1,  # Single-threaded
).prefetch(0)  # NO PREFETCH to save memory
replay_iterator = iter(dataset)

# Initial data collection using agent.collect_policy to populate the buffer
collect_policy = agent.collect_policy
initial_collect_steps = 100  # DRASTICALLY REDUCED from 500 for quick test
collect_driver = dynamic_step_driver.DynamicStepDriver(
    train_env,
    collect_policy,
    observers=[replay_buffer.add_batch],
    num_steps=initial_collect_steps,
)
# Run collection (this will step the environment and add to buffer)
print('Starting initial collection with 100 steps (minimal memory)...')
final_time_step, _ = collect_driver.run()
print('Initial collection finished. Buffer size (frames):', replay_buffer.num_frames())

2025-12-02 15:23:08.605614: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 504000000 exceeds 10% of free system memory.


Starting initial collection with 100 steps (minimal memory)...


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

necesitamos intentar con otras depdencias para resolver los errores y discrepecnias de API/gym y GYMNASIUM para entornos de ATARI

In [ ]:
# Training loop (MINIMAL CPU-only version)

def compute_avg_return(environment, policy, num_episodes=1):
    """Quick evaluation: only 1 episode to save time."""
    total_return = 0.0
    for _ in range(num_episodes):
        time_step = environment.reset()
        episode_return = 0.0
        max_steps = 500  # Limit episode length to avoid long runs
        step_count = 0
        while not time_step.is_last() and step_count < max_steps:
            action_step = policy.action(time_step)
            time_step = environment.step(action_step.action)
            episode_return += time_step.reward.numpy()
            step_count += 1
        total_return += episode_return
    return total_return / num_episodes

num_iterations = 500  # DRASTICALLY REDUCED from 5000 for quick test
collect_steps_per_iteration = 1
batch_size = 16  # Match dataset batch size
log_interval = 50  # Log every 50 steps
eval_interval = 250  # Evaluate only twice during training

# Reset training counter
agent.train_step_counter.assign(0)

print('Starting minimal training loop (500 iterations, CPU-only)...\n')
for iteration in range(num_iterations):
    # Collect one step
    collect_driver = dynamic_step_driver.DynamicStepDriver(
        train_env,
        agent.collect_policy,
        observers=[replay_buffer.add_batch],
        num_steps=collect_steps_per_iteration,
    )
    collect_driver.run()

    # Train step (only if buffer has enough data)
    if replay_buffer.num_frames() > 50:  # Wait for minimum buffer fill
        experience, _ = next(replay_iterator)
        train_loss = agent.train(experience).loss
        step = agent.train_step_counter.numpy()

        if step % log_interval == 0:
            print('step = {0}: loss = {1:.5f}'.format(step, train_loss))

        if step % eval_interval == 0:
            avg_return = compute_avg_return(eval_env, agent.policy, num_episodes=1)
            print('  Evaluation - Average Return = {0:.2f}\n'.format(avg_return))

print('Training complete!')